# LAB 09 — Tokenizer, BIO và Label Alignment
Notebook này gồm: BIO tagging (Calculation 1, 2), subword tokenization + label alignment (Strategy A/B, IGNORE index), shape (Calculation 3) và **Experiment 1 — Tokenization Analysis**.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForTokenClassification
from ner_utils import *
from ner_model import MODEL_NAME
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
ds = load_conll()
print("tokenizer:", MODEL_NAME, "| fast:", tok.is_fast, "| vocab:", tok.vocab_size)
print({k: len(v) for k, v in ds.items()})

tokenizer: distilbert-base-cased | fast: True | vocab: 28996
{'train': 14041, 'validation': 3250, 'test': 3453}


## 1. BIO tagging — Calculation Exercise 1 và 2
`words_to_bio` biến annotation theo cụm từ thành nhãn BIO theo word (đối chiếu với phần tính tay trong `calculations.md`).

In [2]:
words = "Elon Musk founded SpaceX in California .".split()
print(pd.DataFrame({"word": words, "BIO": words_to_bio(words, [("Elon Musk","PER"),("SpaceX","ORG"),("California","LOC")])}).T.to_string(header=False))
w2 = "University of California Berkeley".split()
print()
print(pd.DataFrame({"word": w2, "BIO": words_to_bio(w2, [("University of California Berkeley","ORG")])}).T.to_string(header=False))
# Vì sao không gán tất cả là ORG: hai entity ORG liền nhau sẽ không tách được
print("\nKhông có B/I:", ["I-ORG","I-ORG"], "->", sorted(spans_from_tags(["I-ORG","I-ORG"])), "(1 entity, không phân biệt được 2 tổ chức)")
print("Có B/I      :", ["B-ORG","B-ORG"], "->", sorted(spans_from_tags(["B-ORG","B-ORG"])), "(2 entity)")

word   Elon   Musk  founded  SpaceX  in  California  .
BIO   B-PER  I-PER        O   B-ORG   O       B-LOC  O

word  University     of  California  Berkeley
BIO        B-ORG  I-ORG       I-ORG     I-ORG

Không có B/I: ['I-ORG', 'I-ORG'] -> [('ORG', 0, 2)] (1 entity, không phân biệt được 2 tổ chức)
Có B/I      : ['B-ORG', 'B-ORG'] -> [('ORG', 0, 1), ('ORG', 1, 2)] (2 entity)


## 2. CoNLL-2003: phân bố nhãn và kiểm tra BIO

In [3]:
from collections import Counter
cnt = Counter(l for ex in ds["train"] for l in tags_to_labels(ex["ner_tags"]))
print(pd.Series(cnt).reindex(LABELS).to_frame("train tokens").assign(pct=lambda d: (100*d["train tokens"]/d["train tokens"].sum()).round(2)))
ent_train = Counter(t for ex in ds["train"] for (t,_,_) in spans_from_tags(tags_to_labels(ex["ner_tags"])))
ent_test = Counter(t for ex in ds["test"] for (t,_,_) in spans_from_tags(tags_to_labels(ex["ner_tags"])))
print("\n#entity train:", dict(ent_train), "\n#entity test :", dict(ent_test))
bad = 0
for ex in ds["train"]:
    prev = "O"
    for l in tags_to_labels(ex["ner_tags"]):
        bad += l.startswith("I-") and prev[2:] != l[2:]
        prev = l
print("I- không đứng sau B-/I- cùng type (train):", bad)

        train tokens    pct
O             169578  83.28
B-PER           6600   3.24
I-PER           4528   2.22
B-ORG           6321   3.10
I-ORG           3704   1.82
B-LOC           7140   3.51
I-LOC           1157   0.57
B-MISC          3438   1.69
I-MISC          1155   0.57



#entity train: {'MISC': 3438, 'ORG': 6321, 'PER': 6600, 'LOC': 7140} 
#entity test : {'LOC': 1668, 'PER': 1617, 'MISC': 702, 'ORG': 1661}


I- không đứng sau B-/I- cùng type (train): 0


## 3. Subword tokenization và word_ids

In [4]:
def show(words, tags=None):
    enc = tok(words, is_split_into_words=True, truncation=True, padding=False)
    toks = tok.convert_ids_to_tokens(enc["input_ids"])
    wids = enc.word_ids()
    df = pd.DataFrame({"token": toks, "input_id": enc["input_ids"], "attention_mask": enc["attention_mask"], "word_id": wids})
    if tags is not None:
        df["word_label"] = [tags[w] if w is not None else "-" for w in wids]
        a = align_labels_first_subtoken(wids, [LABEL2ID[t] for t in tags])
        b = align_labels_propagate(wids, [LABEL2ID[t] for t in tags])
        df["A: first-subtoken"] = [ID2LABEL[x] if x != IGNORE_INDEX else "IGNORE(-100)" for x in a]
        df["B: propagate"] = [ID2LABEL[x] if x != IGNORE_INDEX else "IGNORE(-100)" for x in b]
    return df
w = "Barack Obama visited Paris .".split()
print(show(w, ["B-PER","I-PER","O","B-LOC","O"]).to_string())
w = "Nikolaus van der Pas visited Wolverhampton Wanderers in Reykjavik .".split()
t = ["B-PER","I-PER","I-PER","I-PER","O","B-ORG","I-ORG","O","B-LOC","O"]
print(); print(show(w, t).to_string())

     token  input_id  attention_mask  word_id word_label A: first-subtoken  B: propagate
0    [CLS]       101               1      NaN          -      IGNORE(-100)  IGNORE(-100)
1   Barack     14319               1      0.0      B-PER             B-PER         B-PER
2    Obama      7661               1      1.0      I-PER             I-PER         I-PER
3  visited      3891               1      2.0          O                 O             O
4    Paris      2123               1      3.0      B-LOC             B-LOC         B-LOC
5        .       119               1      4.0          O                 O             O
6    [SEP]       102               1      NaN          -      IGNORE(-100)  IGNORE(-100)

            token  input_id  attention_mask  word_id word_label A: first-subtoken  B: propagate
0           [CLS]       101               1      NaN          -      IGNORE(-100)  IGNORE(-100)
1          Nikola     28010               1      0.0      B-PER             B-PER         B-PER

**Strategy được chọn: A (first-subtoken, các subword còn lại = IGNORE).** Lý do và phân tích ảnh hưởng nằm ở `calculations.md` / `reflection.md`; tóm tắt: một word chỉ cần *một* dự đoán; khi suy luận lấy dự đoán của subword đầu nên huấn luyện và suy luận khớp nhau; không phải quyết định gán `I-X` cho mảnh sau (với Strategy B, mảnh sau của word `B-X` bị buộc thành `I-X` dù chưa chắc đó là ý nghĩa đúng).

### IGNORE index trong loss
Token có label = -100 **vẫn đi qua Transformer** (vẫn là input, vẫn tham gia attention của token khác); chỉ là *label của nó không vào loss*.

In [5]:
torch.manual_seed(0)
logits = torch.randn(1, 4, 9)                         # 4 token, 9 nhãn
labels = torch.tensor([[3, -100, 0, 7]])              # token thứ 2 là IGNORE
lossf = torch.nn.CrossEntropyLoss(ignore_index=-100, reduction="none")
per_tok = lossf(logits.view(-1, 9), labels.view(-1))
print("loss từng token:", per_tok.tolist())
print("mean loss (PyTorch, ignore_index):", torch.nn.CrossEntropyLoss(ignore_index=-100)(logits.view(-1, 9), labels.view(-1)).item())
print("mean thủ công trên 3 token hợp lệ :", per_tok[labels.view(-1) != -100].mean().item())
logits.requires_grad_(True)
torch.nn.CrossEntropyLoss(ignore_index=-100)(logits.view(-1, 9), labels.view(-1)).backward()
print("|grad| tại logits của token IGNORE:", logits.grad[0, 1].abs().sum().item())

loss từng token: [2.589226245880127, 0.0, 2.008582592010498, 4.151381492614746]
mean loss (PyTorch, ignore_index): 2.9163968563079834
mean thủ công trên 3 token hợp lệ : 2.9163968563079834
|grad| tại logits của token IGNORE: 0.0


## 4. Calculation Exercise 3 — Shape (kiểm tra bằng model thật)
`B=8, T=128, d=768, C=9`

In [6]:
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=9)
model.eval()
B, T = 8, 128
ids = torch.randint(1000, 5000, (B, T)); att = torch.ones(B, T, dtype=torch.long)
with torch.no_grad():
    hs = model.distilbert(input_ids=ids, attention_mask=att).last_hidden_state
    lg = model(input_ids=ids, attention_mask=att).logits
print("hidden states:", tuple(hs.shape)); print("logits       :", tuple(lg.shape)); print("labels       :", (B, T))
print("classifier   :", model.classifier, "-> W:", tuple(model.classifier.weight.shape), "(C x d), params:", sum(p.numel() for p in model.classifier.parameters()))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1448.57it/s]


[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


hidden states: (8, 128, 768)
logits       : (8, 128, 9)
labels       : (8, 128)
classifier   : Linear(in_features=768, out_features=9, bias=True) -> W: (9, 768) (C x d), params: 6921


## 5. Experiment 1 — Tokenization Analysis
Chọn 30 câu từ test set (15 ngắn: 3–8 word, 15 dài: ≥ 25 word, lấy ngẫu nhiên với seed 42, không có `-DOCSTART-`). Thống kê `# words`, `# tokens` (không tính [CLS]/[SEP]), `# subword splits` (số word bị tách thành ≥ 2 subword) và
`split ratio = #tokens / #words`.

In [7]:
rng = np.random.RandomState(SEED)
test = [ex for ex in ds["test"] if ex["tokens"][0] != "-DOCSTART-"]
short_idx = [i for i, ex in enumerate(test) if 3 <= len(ex["tokens"]) <= 8]
long_idx = [i for i, ex in enumerate(test) if len(ex["tokens"]) >= 25]
sel = list(rng.choice(short_idx, 15, replace=False)) + list(rng.choice(long_idx, 15, replace=False))

def tok_stats(words, tags):
    enc = tok(words, is_split_into_words=True, add_special_tokens=False)
    wids = enc.word_ids()
    n_tok = len(wids)
    pieces = np.bincount(wids, minlength=len(words))          # số subword mỗi word
    split = pieces > 1
    is_ent = np.array([t != "O" for t in tags])
    return dict(words=len(words), tokens=n_tok, splits=int(split.sum()), ratio=n_tok/len(words),
                ent_words=int(is_ent.sum()), ent_split=int((split & is_ent).sum()),
                non_words=int((~is_ent).sum()), non_split=int((split & ~is_ent).sum()))

rows = []
for i in sel:
    ex = test[i]; tags = tags_to_labels(ex["ner_tags"])
    r = tok_stats(ex["tokens"], tags); r["group"] = "short" if len(ex["tokens"]) <= 8 else "long"
    r["text"] = " ".join(ex["tokens"])[:70]; rows.append(r)
df = pd.DataFrame(rows)
print(df[["group","words","tokens","splits","ratio","ent_words","ent_split","text"]].round(3).to_string())

    group  words  tokens  splits  ratio  ent_words  ent_split                                                                    text
0   short      8       9       1  1.125          1          1                                               Partick 16 6 6 4 23 16 24
1   short      7       8       1  1.143          1          1                                  Meyers was not available for comment .
2   short      4       4       0  1.000          0          0                                                     1996 1995 1996 1995
3   short      8       8       0  1.000          0          0                                      The election must be held by May .
4   short      7      11       3  1.571          3          1                                           9. Xu Nannan ( China ) 152.08
5   short      7      23       5  3.286          1          1                              SOCCER - GERMAN FIRST DIVISION SUMMARIES .
6   short      8       8       0  1.000          1          0 

In [8]:
g = df.groupby("group").agg(sent=("words","size"), words=("words","sum"), tokens=("tokens","sum"), splits=("splits","sum"),
                           ent_words=("ent_words","sum"), ent_split=("ent_split","sum"), non_words=("non_words","sum"), non_split=("non_split","sum"))
g["split_ratio"] = g["tokens"]/g["words"]; g["entity_split_rate"] = g["ent_split"]/g["ent_words"]; g["non_entity_split_rate"] = g["non_split"]/g["non_words"]
print("30 câu đã chọn, gộp theo nhóm:\n", g.round(3).to_string())
tot = df[["words","tokens","splits","ent_words","ent_split","non_words","non_split"]].sum()
print("\nTổng 30 câu: split ratio = %.3f | entity-word split rate = %.3f (%d/%d) | non-entity split rate = %.3f (%d/%d)" % (
      tot.tokens/tot.words, tot.ent_split/tot.ent_words, tot.ent_split, tot.ent_words, tot.non_split/tot.non_words, tot.non_split, tot.non_words))
df.to_csv("tokenization_stats.csv", index=False)

30 câu đã chọn, gộp theo nhóm:
        sent  words  tokens  splits  ent_words  ent_split  non_words  non_split  split_ratio  entity_split_rate  non_entity_split_rate
group                                                                                                                                
long     15    538     629      57         73         25        465         32        1.169              0.342                  0.069
short    15     94     145      24         17          7         77         17        1.543              0.412                  0.221

Tổng 30 câu: split ratio = 1.225 | entity-word split rate = 0.356 (32/90) | non-entity split rate = 0.090 (49/542)


### Toàn bộ test set: loại word nào bị tách thành nhiều subword?

In [9]:
recs = []
for ex in test:
    words, tags = ex["tokens"], tags_to_labels(ex["ner_tags"])
    enc = tok(words, is_split_into_words=True, add_special_tokens=False)
    pieces = np.bincount(enc.word_ids(), minlength=len(words))
    for w, t, p in zip(words, tags, pieces):
        recs.append((w, t, p))
W = pd.DataFrame(recs, columns=["word", "tag", "pieces"])
W["split"] = W.pieces > 1
W["type"] = W.tag.map(lambda t: "O" if t == "O" else t[2:])
W["cat"] = np.select([W.word.str.fullmatch(r"[^\w\s]+"), W.word.str.contains(r"\d"), W.word.str.istitle(), W.word.str.isupper(), W.word.str.islower()],
                     ["punctuation", "has digit", "Capitalized", "ALL CAPS", "lowercase"], "other")
W["len_bucket"] = pd.cut(W.word.str.len(), [0, 3, 6, 9, 100], labels=["1-3", "4-6", "7-9", "10+"])
print("Tổng #words = %d, #tokens = %d, split ratio = %.3f, split rate = %.3f" % (len(W), W.pieces.sum(), W.pieces.sum()/len(W), W.split.mean()))
by_type = W.groupby("type").agg(words=("word","size"), split_rate=("split","mean"), avg_pieces=("pieces","mean")).round(3).loc[["O","PER","ORG","LOC","MISC"]]
by_cat = W.groupby("cat").agg(words=("word","size"), split_rate=("split","mean"), avg_pieces=("pieces","mean")).round(3).sort_values("split_rate", ascending=False)
by_len = W.groupby("len_bucket", observed=True).agg(words=("word","size"), split_rate=("split","mean"), avg_pieces=("pieces","mean")).round(3)
print("\n", by_type.to_string(), "\n\n", by_cat.to_string(), "\n\n", by_len.to_string())
print("\nVí dụ word bị tách nhiều mảnh nhất:")
top = W[W.pieces >= 4].drop_duplicates("word").sort_values("pieces", ascending=False).head(12)
for w, t, p in top[["word","tag","pieces"]].itertuples(index=False):
    print(f"  {w:<22}{t:<7}{p}  {tok.tokenize(w)}")

Tổng #words = 46435, #tokens = 63461, split ratio = 1.367, split rate = 0.178

       words  split_rate  avg_pieces
type                               
O     38323       0.120       1.251
PER    2773       0.599       2.055
ORG    2496       0.452       1.936
LOC    1925       0.312       1.775
MISC    918       0.312       1.730 

              words  split_rate  avg_pieces
cat                                       
other           95       0.895       3.674
ALL CAPS      1825       0.859       3.363
has digit     5883       0.339       1.819
Capitalized   9056       0.331       1.568
lowercase    23706       0.066       1.102
punctuation   5870       0.012       1.014 

             words  split_rate  avg_pieces
len_bucket                               
1-3         21640       0.069       1.081
4-6         14460       0.222       1.402
7-9          8000       0.303       1.716
10+          2335       0.497       2.597

Ví dụ word bị tách nhiều mảnh nhất:
  http://www.pt.lu/infoweb/kr

In [10]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
by_type["split_rate"].plot.bar(ax=ax[0], color="#4c78a8"); ax[0].set_title("Split rate theo loại word"); ax[0].set_ylabel("P(word bị tách >=2 subword)")
by_cat["split_rate"].plot.bar(ax=ax[1], color="#f58518"); ax[1].set_title("Split rate theo dạng chữ")
by_len["split_rate"].plot.bar(ax=ax[2], color="#54a24b"); ax[2].set_title("Split rate theo độ dài word (ký tự)")
for a in ax: a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig("tokenization_analysis.png", dpi=130); plt.show()

In [11]:
L = [len(ex["tokens"]) for ex in test]
Tn = [len(tok(ex["tokens"], is_split_into_words=True)["input_ids"]) for ex in test]
print("độ dài câu (word): mean %.1f, max %d | độ dài (subword + special): mean %.1f, max %d, >128: %d" % (np.mean(L), max(L), np.mean(Tn), max(Tn), sum(t > 128 for t in Tn)))

độ dài câu (word): mean 13.4, max 124 | độ dài (subword + special): mean 20.4, max 148, >128: 2


### Nhận xét Experiment 1 (từ các số liệu ở trên)
- **Câu ngắn vs dài (30 câu mẫu):** câu ngắn có split ratio *cao hơn* (1.543) so với câu dài (1.169), trái với trực giác "câu dài nhiều subword hơn". Lý do: câu ngắn trong CoNLL phần lớn là tiêu đề viết HOA (`SOCCER - GERMAN FIRST DIVISION SUMMARIES .` = 7 word nhưng 23 token) và dòng bảng số liệu (`Durum 129.0 1515.9 ...`); số liệu/HOA bị tách rất nhiều. Câu dài là văn xuôi thường nên ít bị tách. Cả 30 câu: split ratio 1.225.
- **Entity vs non-entity:** trong 30 câu, 35.6% word thuộc entity bị tách (32/90) so với 9.0% word non-entity (49/542). Trên toàn test set: PER 59.9%, ORG 45.2%, LOC 31.2%, MISC 31.2%, so với 12.0% của `O` ⇒ entity (tên riêng) bị tách nhiều gấp ~3-5 lần.
- **Loại word bị tách nhiều subword:** (1) từ viết HOA toàn bộ (85.9% bị tách, trung bình 3.4 mảnh); (2) từ dạng số/mã/URL/số điện thoại (`+56-2-699-5595` = 11 mảnh); (3) từ chứa gạch nối (`FOOTBALL-THURSDAY` = 12 mảnh); (4) tên riêng ngoại lai/hiếm (`Reykjavik`, `Nikolaus`); (5) từ dài: tỷ lệ tách tăng từ 6.9% (1-3 ký tự) lên 49.7% (từ ≥ 10 ký tự). Từ thường viết thường chỉ bị tách 6.6%, dấu câu 1.2%. Nguyên nhân: vocab WordPiece cased (~29k) chứa từ phổ biến dạng thường; chữ HOA và tên riêng hiếm không có trong vocab nên bị chia thành mảnh ngắn.
- **Ảnh hưởng đến NER:** toàn test set có 46,435 word nhưng 63,461 subword (split ratio 1.367, 17.8% word bị tách); 26.8% subword nhận nhãn IGNORE (Strategy A). Entity càng hiếm/dài càng nhiều mảnh nên càng cần alignment đúng; chỉ 2/3,453 câu vượt 128 token.
